# Zero-Inflated SAR Negative Binomial Estimation

This notebook demonstrates the **ZINB-SAR** model in `bayespecon`, which jointly models:

1. **Selection equation** — whether an observation is "active" (structural-form SAR-logit with Pólya–Gamma Gibbs)
2. **Count equation** — the intensity of active observations (reduced-form SAR-NB with Pólya–Gamma Gibbs)
3. **Zero allocation** — latent indicators that decompose observed zeros into structural vs. sampling zeros

The model is fit with a 9-block Gibbs sampler — no NUTS required.

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
from bayespecon.models.cross_section.zinb import ZINBSAR

from bayespecon.dgp.zinb import simulate_sar_zinb

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Simulate Data from the ZINB DGP

The `simulate_sar_zinb` function generates data from a two-equation spatial model:
- **Selection**: SAR-logit with parameter λ and covariates Z
- **Count**: Reduced-form SAR-NB with parameter ρ, covariates X, and dispersion α

We use a 10×10 grid (100 observations) for a quick demo.

In [ ]:
# Simulate data from the ZINB DGP
# Use n=50 (50x50 grid → 2500 observations) for reliable parameter recovery.
# Spatial parameters (ρ, λ) require n ≥ 900 for reliable recovery in
# cross-sectional count models. The model raises a UserWarning when n < 900.
data = simulate_sar_zinb(
    n=50,  # 50x50 grid → 2500 observations
    rho=0.4,  # Count SAR parameter
    lam=0.3,  # Selection SAR parameter
    alpha=5.0,  # NB dispersion (larger = less overdispersion)
    beta=np.array([1.0, 0.6]),  # Count coefficients [intercept, slope]
    gamma=np.array([0.3, 1.0]),  # Selection coefficients [intercept, slope]
    target_pi=0.7,  # Target corridor activation probability
    seed=42,
)

y = data["y"]
X = data["X"]
Z = data["Z"]
W_graph = data["W_graph"]
W_sel_graph = data["W_sel_graph"]

print(f"Observations: {len(y)}")
print(f"Zero fraction: {(y == 0).mean():.2%}")
print(f"Mean (y>0): {y[y > 0].mean():.1f}")
print(f"Max: {y.max():.0f}")
print("\nTrue parameters:")
print(f"  ρ (count SAR):  {data['params_true']['rho']}")
print(f"  λ (sel SAR):    {data['params_true']['lam']}")
print(f"  α (dispersion): {data['params_true']['alpha']}")
print(f"  β (count coef): {data['params_true']['beta']}")
print(f"  γ (sel coef):   {data['params_true']['gamma']}")

## 2. Fit the ZINB-SAR Model

The `ZINBSAR` model class accepts:
- `y`: non-negative integer counts
- `X`: count covariate matrix
- `Z`: selection covariate matrix (defaults to `X` if not provided)
- `W`: spatial weights for the count equation
- `W_sel`: spatial weights for the selection equation (defaults to `W`)

The `fit()` method runs a 9-block Gibbs sampler with profile-log-likelihood initialisation.

In [ ]:
# Create and fit the ZINB-SAR model
model = ZINBSAR(
    y=y,
    X=X,
    Z=Z,
    W=W_graph,
    W_sel=W_sel_graph,
)

idata = model.fit(
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=123,
    n_jobs=-1,
    progressbar=True,
)

## 3. Posterior Summary

Compare posterior means to the true parameter values used in the DGP.

In [ ]:
# Posterior summary
summary = az.summary(idata, var_names=["rho", "lam", "alpha", "beta", "gamma"])
print(summary)

# Compare with true values
print("\n--- Parameter Recovery ---")
print(
    f"ρ (count SAR):  true={data['params_true']['rho']:.2f}, "
    f"posterior mean={float(idata.posterior['rho'].mean()):.3f} "
    f"± {float(idata.posterior['rho'].std()):.3f}"
)
print(
    f"λ (sel SAR):    true={data['params_true']['lam']:.2f}, "
    f"posterior mean={float(idata.posterior['lam'].mean()):.3f} "
    f"± {float(idata.posterior['lam'].std()):.3f}"
)
print(
    f"α (dispersion): true={data['params_true']['alpha']:.2f}, "
    f"posterior mean={float(idata.posterior['alpha'].mean()):.3f} "
    f"± {float(idata.posterior['alpha'].std()):.3f}"
)

# Note: spatial parameters (ρ, λ) require large samples (n ≥ 900)
# for reliable recovery in cross-sectional count models. The posterior
# is typically wide and may be attenuated toward zero at moderate n.
# The model raises a UserWarning when n < 900.

## 4. Convergence Diagnostics

Check R-hat and effective sample sizes to verify MCMC convergence.

In [ ]:
# Convergence diagnostics
az.plot_trace(idata, var_names=["rho", "lam", "alpha"], compact=False)
plt.tight_layout()
plt.show()

# R-hat and ESS
diagnostics = az.summary(idata, var_names=["rho", "lam", "alpha"])
print(diagnostics[["r_hat", "ess_bulk", "ess_tail"]])

## 5. Zero Attribution Analysis

The ZINB model decomposes observed zeros into two types:
- **Structural zeros**: the observation is inactive (d=0)
- **Sampling zeros**: the observation is active (d=1) but the NB draw was zero

The `zero_attribution()` method computes the posterior probability of each type for every zero observation.

In [ ]:
# Zero attribution
attribution = model.zero_attribution()

print(f"Number of zero observations: {len(attribution['zero_indices'])}")
print(f"\nMean P(structural | y=0): {attribution['structural_prob'].mean():.3f}")
print(f"Mean P(sampling | y=0):   {attribution['sampling_prob'].mean():.3f}")

# Histogram of structural zero probabilities
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(attribution["structural_prob"], bins=20, edgecolor="white")
ax[0].set_xlabel("P(structural zero | y=0)")
ax[0].set_ylabel("Count")
ax[0].set_title("Structural Zero Probabilities")

ax[1].hist(attribution["sampling_prob"], bins=20, edgecolor="white", color="C1")
ax[1].set_xlabel("P(sampling zero | y=0)")
ax[1].set_ylabel("Count")
ax[1].set_title("Sampling Zero Probabilities")
plt.tight_layout()
plt.show()

## 6. Corridor Activation Probabilities

The `corridor_probabilities()` method returns the posterior-mean probability that each observation is "active" (i.e., in the count regime rather than the always-zero regime).

In [ ]:
# Corridor activation probabilities
pi = model.corridor_probabilities()

print(f"Mean corridor probability: {pi.mean():.3f}")
print(f"Min corridor probability:   {pi.min():.3f}")
print(f"Max corridor probability:   {pi.max():.3f}")

# Compare with true d
d_true = data["d"]
print(f"\nTrue activation rate: {d_true.mean():.3f}")
print(f"Estimated activation rate: {pi.mean():.3f}")

# Scatter: pi vs observed y
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(pi, y, alpha=0.5, s=20)
ax.set_xlabel("Estimated P(active)")
ax.set_ylabel("Observed count y")
ax.set_title("Corridor Activation Probability vs Observed Count")
ax.axhline(0, color="grey", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Fitted Mean Counts

The `_fitted_mean_from_posterior()` method returns E[y_i] = π_i · exp(η_i^cnt), the posterior-mean expected count for each observation.

In [ ]:
# Fitted mean counts
fitted = model._fitted_mean_from_posterior()

print(f"Mean fitted count: {fitted.mean():.2f}")
print(f"Mean observed count: {y.mean():.2f}")

# Scatter: fitted vs observed
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(fitted, y, alpha=0.5, s=20)
ax.plot(
    [0, max(y.max(), fitted.max())],
    [0, max(y.max(), fitted.max())],
    "r--",
    alpha=0.5,
    label="45° line",
)
ax.set_xlabel("Fitted E[y]")
ax.set_ylabel("Observed y")
ax.set_title("Fitted vs Observed Counts")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Posterior Distributions of Key Parameters

Visualize the posterior distributions of the spatial parameters ρ and λ, and the NB dispersion α, with true values marked.

In [ ]:
# Posterior distributions with true values
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ρ (count SAR)
az.plot_posterior(
    idata, var_names=["rho"], ref_val=data["params_true"]["rho"], ax=axes[0]
)
axes[0].set_title("ρ (Count SAR)")

# λ (selection SAR)
az.plot_posterior(
    idata, var_names=["lam"], ref_val=data["params_true"]["lam"], ax=axes[1]
)
axes[1].set_title("λ (Selection SAR)")

# α (NB dispersion)
az.plot_posterior(
    idata, var_names=["alpha"], ref_val=data["params_true"]["alpha"], ax=axes[2]
)
axes[2].set_title("α (NB Dispersion)")

plt.tight_layout()
plt.show()

# Note: spatial parameters (ρ, λ) require large samples (n ≥ 900)
# for reliable recovery in cross-sectional count models. The posterior
# is typically wide and may be attenuated toward zero at moderate n.

## Summary

The ZINB-SAR model in `bayespecon` provides:

1. **Joint estimation** of selection (SAR-logit) and count (SAR-NB) equations
2. **Zero allocation** — posterior decomposition of zeros into structural vs sampling
3. **Different W per equation** — the selection and count processes can operate on different spatial scales
4. **Gibbs-only sampling** — no NUTS required; the 9-block PG-Gibbs sampler is fast and reliable
5. **Profile-log-likelihood initialisation** — smart starting values that avoid the ρ–β multimodality trap

**Caveat**: Spatial parameters (ρ, λ) require large samples for reliable recovery in cross-sectional count models. The model raises a `UserWarning` when n < 900. At moderate sample sizes, the posterior for ρ and λ is typically wide and may be attenuated toward zero. For precise spatial parameter estimation, use n ≥ 900 or panel data.

Key API:
- `ZINBSAR(y, X, Z, W, W_sel)` — model specification
- `.fit(draws, tune, chains)` — Gibbs sampling
- `.corridor_probabilities()` — P(active) for each observation
- `.zero_attribution()` — structural vs sampling zero decomposition
- `._fitted_mean_from_posterior()` — E[y] = π · exp(η^cnt)